In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')


import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

DB_USER     = 'your_username'
DB_PASSWORD = 'your_password'
DB_HOST     = 'localhost'
DB_PORT     = '3306'          
DB_NAME     = 'livestock_db'

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

print(f'Connected to database: {DB_NAME}')

In [ ]:

query = """
    SELECT 
        datetime,
        event,
        temp,
        airTemp,
        humidity,
        accel,
        rumination,
        ageInDays,
        lactationNumber,
        aveHerdTemp,
        aveGroupTemp,
        aveHerdActivity,
        aveGroupActivity,
        aveHerdRumination,
        aveGroupRumination
    FROM livestock_sensor_data
    ORDER BY datetime ASC
"""

df_raw = pd.read_sql(query, engine)

print(f'  Data extracted successfully')
print(f'   Rows    : {df_raw.shape[0]:,}')
print(f'   Columns : {df_raw.shape[1]}')
df_raw.head()

In [ ]:
print('Column Data Types:')
print(df_raw.dtypes)
print('Basic Statistics:')
df_raw.describe()

In [ ]:
df = df_raw.copy()

df['datetime'] = pd.to_datetime(df['datetime'],format='mixed',coerce='error')


df['date'] = df['datetime'].dt.date
df['time'] = df['datetime'].dt.time
df['hour'] = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek

print(f'Datetime parsed | Range: {df["datetime"].min()} → {df["datetime"].max()}')

In [ ]:

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report = missing_report[missing_report['Missing Count'] > 0]

print('Missing Value Report:')
print(missing_report if not missing_report.empty else ' No missing values found')

In [ ]:

# Numeric sensor columns — forward fill (preserves time-series continuity)
sensor_cols = ['temp', 'airTemp', 'humidity', 'accel', 'rumination',
               'aveHerdTemp', 'aveGroupTemp', 'aveHerdActivity',
               'aveGroupActivity', 'aveHerdRumination', 'aveGroupRumination']

df[sensor_cols] = df[sensor_cols].fillna(method='ffill').fillna(method='bfill')

# Categorical — fill with 'None'
df['event'] = df['event'].fillna('None')

print(f'Missing values handled | Remaining nulls: {df.isnull().sum().sum()}')

In [ ]:

df = df.drop_duplicates(subset=['datetime'])
after = len(df)
print(f'Duplicates removed: {before - after} rows dropped')


def remove_outliers_iqr(df, col, factor=3.0):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return df[(df[col] >= lower) & (df[col] <= upper)]


for col in ['temp', 'accel', 'rumination']:
    before = len(df)
    df = remove_outliers_iqr(df, col)
    print(f'   {col}: {before - len(df)} outliers removed')

print(f'\n Final shape after cleaning: {df.shape}')

In [ ]:

# Standard formula used in livestock heat stress research
df['THI'] = (1.8 * df['airTemp'] + 32) - (
    (0.55 - 0.0055 * df['humidity']) * ((1.8 * df['airTemp'] + 32) - 58)
)

# Heat stress classification
df['heat_stress_level'] = pd.cut(
    df['THI'],
    bins=[-np.inf, 68, 72, 80, np.inf],
    labels=['None', 'Mild', 'Moderate', 'Severe']
)

print(' THI calculated and heat stress levels classified')
print(df['heat_stress_level'].value_counts())

In [ ]:
# ── Rolling Averages (24-hour window = 96 records at 15-min intervals) ──
window = 96

df['temp_rolling_24h']       = df['temp'].rolling(window=window, min_periods=1).mean()
df['accel_rolling_24h']      = df['accel'].rolling(window=window, min_periods=1).mean()
df['rumination_rolling_24h'] = df['rumination'].rolling(window=window, min_periods=1).mean()


In [ ]:

df['rumination_per_day'] = df.groupby('date')['rumination'].transform('sum')


df['temp_vs_herd']       = df['temp'] - df['aveHerdTemp']
df['temp_vs_group']      = df['temp'] - df['aveGroupTemp']
df['accel_vs_herd']      = df['accel'] - df['aveHerdActivity']
df['accel_vs_group']     = df['accel'] - df['aveGroupActivity']
df['rum_vs_herd']        = df['rumination'] - df['aveHerdRumination']
df['rum_vs_group']       = df['rumination'] - df['aveGroupRumination']


In [ ]:

df['age_group'] = pd.cut(
    df['ageInDays'],
    bins=[0, 30, 90, 180, 365, np.inf],
    labels=['Calf', 'Heifer', 'Cow']
)
print(df['age_group'].value_counts())

In [ ]:

df['label'] = df['event'].apply(lambda x: 0 if x == 'None' else 1)

print(df['label'].value_counts())
print(f'   Class balance: {df["label"].mean()*100:.1f}% positive (sick)')

In [ ]:
# ── Validation Rules ────────────────────────────────────────────────
validation_errors = []

# 1. Temp within physiological range for cattle (35-42°C)
invalid_temp = df[(df['temp'] < 35) | (df['temp'] > 42)]
if len(invalid_temp) > 0:
    validation_errors.append(f'{len(invalid_temp)} rows with temp outside 35-42°C range')

# 2. Humidity between 0-100%
invalid_humidity = df[(df['humidity'] < 0) | (df['humidity'] > 100)]
if len(invalid_humidity) > 0:
    validation_errors.append(f'{len(invalid_humidity)} rows with invalid humidity')

# 3. No nulls in final dataset
remaining_nulls = df.isnull().sum().sum()
if remaining_nulls > 0:
    validation_errors.append(f'{remaining_nulls} null values remaining')

# 4. Datetime is sorted
if not df['datetime'].is_monotonic_increasing:
    validation_errors.append('Datetime column is not sorted')
    df = df.sort_values('datetime').reset_index(drop=True)

if validation_errors:
    for err in validation_errors:
        print(err)
else:
    print(' All validation checks passed')

In [ ]:

output_path = '../Data/processed_livestock_data.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned data saved to: {output_path}')
print(f'   Final shape: {df.shape}')

In [ ]:
# ── Optional: Write back to SQL ────────────────────────────────────
# Uncomment to load processed data back into a new SQL table

# df.to_sql(
#     name='livestock_processed',
#     con=engine,
#     if_exists='replace',
#     index=False
# )

In [ ]:
print('Final Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:02d}. {col}')

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Label Distribution (0=Healthy, 1=Sick)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Temperature distribution
df['temp'].hist(bins=40, ax=axes[1], color='#3498db', edgecolor='black')
axes[1].set_title('Body Temperature Distribution')
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../Outputs/etl_summary_plots.png', dpi=150, bbox_inches='tight')
plt.show()